# פרויקט לימוד מכונה - חלק ב'

במחברת זו נבנה תהליך עבודה מסודר עבור חלק ב' של הפרויקט: טעינת הנתונים, הכנתם לאימון, חלוקה לסט אימון וסט אימות, ובהמשך אימון והשוואה בין מודלים שונים.

בשלב הנוכחי המחברת כוללת את שלד העבודה ואת שלב טעינת והכנת הנתונים בלבד. סעיפי המודלים מופיעים ככותרות להמשך, אך עדיין לא ממומשים.

## 1. ייבוא ספריות והגדרות ראשוניות

נייבא ספריות בסיסיות הדרושות לטעינת הנתונים, בדיקה ראשונית וחלוקה לסט אימון וסט אימות. בהמשך נוסיף ספריות נוספות רק כאשר נגיע לסעיפי המודלים.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


RANDOM_STATE = 42

## 2. טעינת קבצי הנתונים

נטען את קובץ האימון, קובץ המבחן הסופי וקובץ הדוגמה להגשה. בשלב זה קובץ המבחן הסופי משמש רק לבדיקת מבנה, ולא לבחירת מודלים או לכוונון שלהם.

In [ ]:
BASE_DIR = Path.cwd()

TRAIN_FILE = BASE_DIR / "Xy_train.csv"
TEST_FILE = BASE_DIR / "X_test.csv"
EXAMPLE_SUBMISSION_FILE = BASE_DIR / "y test example.xlsx"

required_files = [TRAIN_FILE, TEST_FILE, EXAMPLE_SUBMISSION_FILE]
missing_files = [file_path.name for file_path in required_files if not file_path.exists()]

if missing_files:
    raise FileNotFoundError(f"הקבצים הבאים חסרים בתיקיית העבודה: {missing_files}")

train_raw = pd.read_csv(TRAIN_FILE)
test_raw = pd.read_csv(TEST_FILE)
example_submission = pd.read_excel(EXAMPLE_SUBMISSION_FILE)

display(train_raw.head())
display(test_raw.head())
display(example_submission.head())

## 3. בדיקות מבנה בסיסיות

נבדוק את גודל הקבצים, שמות העמודות, ערכים חסרים וכפילויות. בדיקות אלה נועדו לוודא שאנחנו עובדים עם הקבצים הנכונים לפני שמתחילים לבנות מודלים.

In [ ]:
data_summary = pd.DataFrame(
    {
        "קובץ": ["אימון", "מבחן סופי", "דוגמת הגשה"],
        "מספר שורות": [len(train_raw), len(test_raw), len(example_submission)],
        "מספר עמודות": [train_raw.shape[1], test_raw.shape[1], example_submission.shape[1]],
        "מספר ערכים חסרים": [
            int(train_raw.isna().sum().sum()),
            int(test_raw.isna().sum().sum()),
            int(example_submission.isna().sum().sum()),
        ],
        "מספר שורות כפולות": [
            int(train_raw.duplicated().sum()),
            int(test_raw.duplicated().sum()),
            int(example_submission.duplicated().sum()),
        ],
    }
)

display(data_summary)

In [ ]:
columns_summary = pd.DataFrame(
    {
        "עמודות בקובץ האימון": pd.Series(train_raw.columns),
        "עמודות בקובץ המבחן הסופי": pd.Series(test_raw.columns),
        "עמודות בקובץ הדוגמה": pd.Series(example_submission.columns),
    }
)

display(columns_summary)

In [ ]:
missing_summary = pd.DataFrame(
    {
        "חסרים באימון": train_raw.isna().sum(),
        "חסרים במבחן הסופי": test_raw.isna().sum(),
    }
).fillna(0).astype(int)

display(missing_summary)

## 4. זיהוי עמודת המטרה והפרדת מאפיינים

נזהה את עמודת המטרה מתוך קובץ האימון בפועל. לא נניח מראש את שם העמודה או את צורת הכתיבה שלה. לאחר הזיהוי נפריד בין מאפייני הקלט לבין משתנה המטרה.

In [ ]:
train_columns = list(train_raw.columns)
test_columns = list(test_raw.columns)

columns_only_in_train = [column for column in train_columns if column not in test_columns]

if len(columns_only_in_train) != 1:
    raise ValueError(
        "לא ניתן לזהות באופן חד-משמעי את עמודת המטרה. "
        f"עמודות שמופיעות רק באימון: {columns_only_in_train}"
    )

target_column = columns_only_in_train[0]
feature_columns = [column for column in train_columns if column != target_column]

missing_in_test = [column for column in feature_columns if column not in test_columns]
extra_in_test = [column for column in test_columns if column not in feature_columns]

if missing_in_test or extra_in_test:
    raise ValueError(
        "מבנה עמודות המאפיינים באימון ובמבחן הסופי אינו תואם. "
        f"חסרות במבחן: {missing_in_test}; עודפות במבחן: {extra_in_test}"
    )

X = train_raw[feature_columns].copy()
y = train_raw[target_column].copy()
X_test_final = test_raw[feature_columns].copy()

print(f"עמודת המטרה שזוהתה: {target_column}")
print(f"מספר מאפיינים: {len(feature_columns)}")

In [ ]:
target_distribution = (
    y.value_counts(dropna=False)
    .rename_axis("ערך המטרה")
    .reset_index(name="מספר רשומות")
)
target_distribution["אחוז"] = (target_distribution["מספר רשומות"] / len(y) * 100).round(2)

display(target_distribution)

## 5. ניקוי נתונים לפי החלטות חלק א'

ניישם בתוך המחברת את לוגיקת הניקוי המתוקנת מחלק א'. קבצי המקור נשארים ללא שינוי, והטבלאות הנקיות נשמרות בזיכרון המחברת בלבד.

בקובץ האימון מותר להסיר רשומות לא תקינות כאשר זה נעשה במחברת המתוקנת. בקובץ המבחן הסופי לא נמחק רשומות ולא נשנה את סדר הרשומות, כדי שקובץ החיזויים הסופי יתאים בדיוק לקובץ המקור.

רשומות ללא ערך במשתנה המטרה אינן יכולות לשמש ללמידה מפוקחת, ולכן נסיר אותן רק מנתוני האימון. לקובץ המבחן הסופי אין משתנה מטרה, ולכן לעולם לא נסיר ממנו רשומות.

In [ ]:
service_columns = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Seat comfort",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Inflight service",
    "Cleanliness",
]

numeric_like_columns = [
    "Age",
    "Flight Distance",
    "Plane colors",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes",
    *service_columns,
]

engineered_features = ["Age_Category", "Flight_Type", "Total_Service_Score"]


def apply_corrected_part_a_cleaning(df_to_clean, is_train=True, target_column_name=None):
    df_clean = df_to_clean.copy()
    original_index = df_clean.index.copy()
    original_rows = len(df_clean)
    summary = {"מספר שורות לפני ניקוי": original_rows}

    for column in numeric_like_columns:
        if column in df_clean.columns:
            df_clean[column] = pd.to_numeric(df_clean[column], errors="coerce")

    # 1. זיהוי והסרת ערכי Class לא תקינים באימון בלבד
    removed_invalid_class_rows = 0
    removed_missing_target_rows = 0
    if "Class" in df_clean.columns:
        valid_classes = ["Eco", "Eco Plus", "Business", "Unknown"]
        missing_class_mask = df_clean["Class"].isna()
        invalid_class_mask = (~df_clean["Class"].isin(valid_classes)) & df_clean["Class"].notna()
        summary["ערכי Class חסרים שהוחלפו"] = int(missing_class_mask.sum())
        summary["ערכי Class לא תקינים"] = int(invalid_class_mask.sum())

        if is_train:
            removed_invalid_class_rows = int(invalid_class_mask.sum())
            df_clean = df_clean.drop(index=df_clean.index[invalid_class_mask]).copy()
        else:
            df_clean.loc[invalid_class_mask, "Class"] = np.nan

        # 2. החלפת Class חסר ו-Unknown ב-Business
        missing_class_mask = df_clean["Class"].isna()
        unknown_mask = df_clean["Class"] == "Unknown"
        summary["ערכי Unknown ב-Class שהוחלפו"] = int(unknown_mask.sum())
        df_clean.loc[missing_class_mask | unknown_mask, "Class"] = "Business"

    # 3. טיפול בערכי Gate location חריגים והשלמה לפי חציון
    if "Gate location" in df_clean.columns:
        invalid_gate_mask = (~df_clean["Gate location"].between(1, 5)) & df_clean["Gate location"].notna()
        summary["ערכי Gate location חריגים"] = int(invalid_gate_mask.sum())
        df_clean.loc[invalid_gate_mask, "Gate location"] = np.nan
        gate_median = df_clean["Gate location"].median()
        df_clean["Gate location"] = df_clean["Gate location"].fillna(gate_median)
        summary["חציון Gate location להשלמה"] = gate_median

    # 4. השלמת Leg room service לפי Class + Type of Travel, אחר כך Class, אחר כך חציון כללי
    if {"Leg room service", "Class", "Type of Travel"}.issubset(df_clean.columns):
        leg_missing_before = int(df_clean["Leg room service"].isna().sum())
        group_median_1 = df_clean.groupby(["Class", "Type of Travel"])["Leg room service"].transform("median")
        group_median_2 = df_clean.groupby("Class")["Leg room service"].transform("median")
        global_median = df_clean["Leg room service"].median()

        df_clean["Leg room service"] = df_clean["Leg room service"].fillna(group_median_1)
        df_clean["Leg room service"] = df_clean["Leg room service"].fillna(group_median_2)
        df_clean["Leg room service"] = df_clean["Leg room service"].fillna(global_median)

        summary["חסרי Leg room service לפני השלמה"] = leg_missing_before
        summary["חציון כללי Leg room service"] = global_median

    # 5. טיפול בערכי Age חריגים והשלמה לפי חציון
    if "Age" in df_clean.columns:
        invalid_age_mask = ((df_clean["Age"] < 0) | (df_clean["Age"] > 110)) & df_clean["Age"].notna()
        summary["ערכי Age חריגים"] = int(invalid_age_mask.sum())
        df_clean.loc[invalid_age_mask, "Age"] = np.nan
        age_median = df_clean["Age"].median()
        df_clean["Age"] = df_clean["Age"].fillna(age_median)
        summary["חציון Age להשלמה"] = age_median

    # 6. טיפול בערכי Flight Distance שליליים והשלמה היררכית
    if {"Flight Distance", "Class", "Type of Travel"}.issubset(df_clean.columns):
        negative_distance_mask = (df_clean["Flight Distance"] < 0) & df_clean["Flight Distance"].notna()
        summary["ערכי Flight Distance שליליים"] = int(negative_distance_mask.sum())
        df_clean.loc[negative_distance_mask, "Flight Distance"] = np.nan

        distance_median_group_1 = df_clean.groupby(["Class", "Type of Travel"])["Flight Distance"].transform("median")
        distance_median_group_2 = df_clean.groupby("Class")["Flight Distance"].transform("median")
        distance_global_median = df_clean["Flight Distance"].median()

        df_clean["Flight Distance"] = df_clean["Flight Distance"].fillna(distance_median_group_1)
        df_clean["Flight Distance"] = df_clean["Flight Distance"].fillna(distance_median_group_2)
        df_clean["Flight Distance"] = df_clean["Flight Distance"].fillna(distance_global_median)
        summary["חציון כללי Flight Distance"] = distance_global_median

    # 7. הסרת Plane colors מתוך הטבלה שכבר נוקתה, בלי לחזור לטבלת המקור
    if "Plane colors" in df_clean.columns:
        df_clean = df_clean.drop(columns=["Plane colors"])
        summary["Plane colors הוסר"] = True
    else:
        summary["Plane colors הוסר"] = False

    # לפני השלמה כללית, מסירים באימון בלבד רשומות ללא משתנה מטרה
    if is_train and target_column_name is not None and target_column_name in df_clean.columns:
        missing_target_mask = df_clean[target_column_name].isna()
        removed_missing_target_rows = int(missing_target_mask.sum())
        if removed_missing_target_rows > 0:
            df_clean = df_clean.drop(index=df_clean.index[missing_target_mask]).copy()

    # 8. השלמת חסרים מספריים שנותרו לפי חציון
    numeric_cols = [
        col for col in df_clean.select_dtypes(include=[np.number]).columns
        if col != target_column_name
    ]
    for col in numeric_cols:
        if df_clean[col].isna().any():
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())

    # 9. השלמת חסרים קטגוריאליים שנותרו לפי שכיח
    categorical_cols = [
        col for col in df_clean.select_dtypes(include=["object"]).columns
        if col != target_column_name
    ]
    for col in categorical_cols:
        if df_clean[col].isna().any():
            df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

    # 10. יצירת מאפיינים חדשים כמו במחברת המתוקנת
    if "Age" in df_clean.columns:
        df_clean["Age_Category"] = pd.cut(
            df_clean["Age"],
            bins=[0, 12, 18, 65, np.inf],
            labels=["Child", "Teen", "Adult", "Senior"],
            include_lowest=True,
        )

    if "Flight Distance" in df_clean.columns:
        df_clean["Flight_Type"] = pd.cut(
            df_clean["Flight Distance"],
            bins=[0, 1000, 3000, df_clean["Flight Distance"].max()],
            labels=["Short-Haul", "Medium-Haul", "Long-Haul"],
            include_lowest=True,
        )

    available_service_columns = [column for column in service_columns if column in df_clean.columns]
    if available_service_columns:
        df_clean["Total_Service_Score"] = df_clean[available_service_columns].mean(axis=1)

    summary["מספר שורות אחרי ניקוי"] = len(df_clean)
    summary["רשומות Class שהוסרו"] = removed_invalid_class_rows
    summary["רשומות שהוסרו בגלל יעד חסר"] = removed_missing_target_rows
    summary["סך ערכים חסרים אחרי ניקוי"] = int(df_clean.isna().sum().sum())
    summary["מאפיינים הנדסיים נוצרו"] = all(feature in df_clean.columns for feature in engineered_features)
    summary["סדר מקורי נשמר"] = df_clean.index.equals(original_index)

    return df_clean, summary


train_for_cleaning = X.copy()
train_for_cleaning[target_column] = y

train_clean, train_cleaning_summary = apply_corrected_part_a_cleaning(
    train_for_cleaning,
    is_train=True,
    target_column_name=target_column,
)

X_test_clean, test_cleaning_summary = apply_corrected_part_a_cleaning(
    X_test_final,
    is_train=False,
    target_column_name=None,
)

X_model = train_clean.drop(columns=[target_column]).copy()
y_model = train_clean[target_column].copy()
X_test_final = X_test_clean.copy()

missing_in_clean_test = [column for column in X_model.columns if column not in X_test_final.columns]
extra_in_clean_test = [column for column in X_test_final.columns if column not in X_model.columns]

if missing_in_clean_test or extra_in_clean_test:
    raise ValueError(
        "מבנה המאפיינים לאחר הניקוי אינו תואם בין אימון למבחן. "
        f"חסרות במבחן: {missing_in_clean_test}; עודפות במבחן: {extra_in_clean_test}"
    )

X_test_final = X_test_final[X_model.columns].copy()

cleaning_summary = pd.DataFrame([train_cleaning_summary, test_cleaning_summary], index=["אימון", "מבחן סופי"])
display(cleaning_summary)

print("הנתונים נוקו בתוך המחברת בלבד. קבצי המקור לא שונו.")

### בדיקות לאחר הניקוי

נציג בדיקות שמוודאות שהניקוי בוצע לפי החלטות חלק א', ושקובץ המבחן הסופי נשמר באותו אורך ובאותו סדר.

In [ ]:
rows_check = pd.DataFrame(
    {
        "קובץ": ["אימון", "מבחן סופי"],
        "שורות לפני ניקוי": [len(train_raw), len(test_raw)],
        "שורות אחרי ניקוי": [len(train_clean), len(X_test_final)],
        "שורות שהוסרו בגלל יעד חסר": [train_cleaning_summary["רשומות שהוסרו בגלל יעד חסר"], 0],
    }
)

display(rows_check)

test_order_preserved = X_test_final.index.equals(test_raw.index)
test_row_count_preserved = len(X_test_final) == len(test_raw)

print(f"מספר שורות בקובץ המבחן נשמר: {test_row_count_preserved}")
print(f"סדר הרשומות בקובץ המבחן נשמר: {test_order_preserved}")
print(f"מספר שורות אימון שהוסרו בגלל יעד חסר: {train_cleaning_summary['רשומות שהוסרו בגלל יעד חסר']}")

missing_after_cleaning = pd.DataFrame(
    {
        "חסרים במאפייני X באימון לאחר ניקוי": X_model.isna().sum(),
        "חסרים במבחן לאחר ניקוי": X_test_final.isna().sum(),
    }
).fillna(0).astype(int)

display(missing_after_cleaning)

missing_y_after_cleaning = int(y_model.isna().sum())
print(f"חסרים ב-y לאחר ניקוי: {missing_y_after_cleaning}")

plane_colors_removed = ("Plane colors" not in train_clean.columns) and ("Plane colors" not in X_test_final.columns)
engineered_features_created = all(
    feature in train_clean.columns and feature in X_test_final.columns
    for feature in engineered_features
)

print(f"Plane colors הוסר מהאימון ומהמבחן: {plane_colors_removed}")
print(f"המאפיינים Age_Category, Flight_Type, Total_Service_Score נוצרו: {engineered_features_created}")

assert test_row_count_preserved
assert test_order_preserved
assert X_model.isna().sum().sum() == 0
assert missing_y_after_cleaning == 0
assert X_test_final.isna().sum().sum() == 0
assert plane_colors_removed
assert engineered_features_created

## 6. הכנת נתונים ראשונית לאימון ולאימות

לאחר ניקוי הנתונים בתוך המחברת, נשתמש בטבלת האימון הנקייה ובטבלת המבחן הנקייה בזיכרון. קובץ המבחן הסופי עדיין לא ישמש לאימון, לבחירת מודלים או לכוונון.

In [ ]:
if y_model.isna().any():
    raise ValueError("נמצאו ערכים חסרים בעמודת המטרה לאחר הניקוי. יש לטפל בכך לפני חלוקת הנתונים.")

original_test_index = X_test_final.index.copy()
original_test_length = len(X_test_final)

print("הנתונים הנקיים מוכנים לחלוקה ראשונית לאימון ולאימות.")

## 7. חלוקה לסט אימון וסט אימות

נחלק את קובץ האימון הנקי לסט אימון ולסט אימות. סט האימות יישמר בצד וישמש להערכת ביצועי המודלים על נתונים שלא שימשו לאימון. קובץ המבחן הסופי נשאר מחוץ לתהליך זה.

In [ ]:
from sklearn.model_selection import train_test_split

stratify_target = y_model if y_model.nunique(dropna=False) > 1 else None

X_train, X_valid, y_train, y_valid = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_target,
)

split_summary = pd.DataFrame(
    {
        "סט": ["אימון", "אימות", "מבחן סופי"],
        "מספר רשומות": [len(X_train), len(X_valid), len(X_test_final)],
        "מספר מאפיינים": [X_train.shape[1], X_valid.shape[1], X_test_final.shape[1]],
    }
)

display(split_summary)

In [ ]:
train_target_distribution = y_train.value_counts(normalize=True, dropna=False).rename("אימון")
valid_target_distribution = y_valid.value_counts(normalize=True, dropna=False).rename("אימות")

split_target_distribution = (
    pd.concat([train_target_distribution, valid_target_distribution], axis=1)
    .fillna(0)
    .mul(100)
    .round(2)
)

display(split_target_distribution)

In [ ]:
assert len(X_test_final) == original_test_length
assert X_test_final.index.equals(original_test_index)

print("בדיקת קובץ המבחן הסופי הסתיימה: מספר השורות והסדר המקורי נשמרו.")

## 8. עצי החלטה

בסעיף זה נבנה בהמשך עץ החלטה מלא, נכוונן היפר-פרמטרים, נציג את העץ הנבחר, ננתח חשיבות משתנים ונעביר רשומת אימות לדוגמה דרך העץ.

## 9. רשתות נוירונים / MLP

בסעיף זה נבנה בהמשך רשת נוירונים, נריץ מודל ברירת מחדל, נכוונן היפר-פרמטרים ונסביר את הקונפיגורציה שנבחרה.

## 10. אשכולות בשיטת K-Means

בסעיף זה נריץ בהמשך K-Means, נבדוק את ההתאמה בין האשכולות למחלקות ונציג גרפים מתאימים להמחשת המבנה שהתקבל.

## 11. מסווג בייסיאני נאיבי

בסעיף זה נאמן בהמשך שני מסווגים בייסיאניים מסוגים שונים, נציג ביצועים ונבדוק הסתברויות למחלקות עבור רשומת אימות אחת.

## 12. השוואה בין מודלים

בסעיף זה נשווה בהמשך בין המודלים המפוקחים וננסח מסקנה לגבי המודל המתאים ביותר למשימת הסיווג.

## 13. המודל הנבחר

בסעיף זה נציג בהמשך את המודל שנבחר להגשה, את הקונפיגורציה שלו ואת מטריצת הבלבול על סט האימות.

## 14. חיזויים סופיים וייצוא קובץ ההגשה

בסעיף זה נאמן בהמשך את המודל הסופי על נתוני האימון המלאים, נחזה את התוויות עבור קובץ המבחן הסופי ונייצא קובץ אקסל בפורמט קובץ הדוגמה.